# NOURA EL KHOLTI

## Q-Learning

### 1. Introduction au Reinforcement Learning

L'apprentissage par renforcement (RL) est une branche de l'intelligence artificielle où un agent apprend à prendre des décisions en interagissant avec un environnement. L'objectif est de maximiser une récompense cumulative au fil du temps. À chaque étape, l'agent observe son état actuel, choisit une action, et reçoit une récompense ainsi qu'un nouvel état.

Le Q-Learning est une méthode "off-policy" qui permet de déterminer une politique optimale en apprenant les valeurs de la fonction $Q(s, a)$, représentant la qualité d'une action $a$ dans un état $s$.
La mise à jour de la Q-Table repose sur l'équation de Bellman :
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$
- $\alpha$ (Alpha) : Taux d'apprentissage (Learning Rate).
- $\gamma$ (Gamma) : Facteur de réduction (Discount Factor).
- $r$ : Récompense immédiate reçue.
- $\max Q(s', a')$ : Estimation de la valeur maximale du futur état.

### 2. Modélisation de l'Environnement (Le Robot Logistique)

Nous modélisons un robot de transport dans un entrepôt composé de 4 zones clés:
1. Zone de Stockage (Départ fréquent)
2. Zone de Tri
3. Zone d'Emballage
4. Quai d'Expédition (Objectif final)

Paramètres de simulation:
- Nombre d'états : 4
- Actions possibles : Se déplacer vers une zone adjacente.
- Récompense : 100 si l'agent atteint le Quai d'Expédition, 0 sinon.

### 3. Implémentation

Configuration de l'Environnement

In [ ]:
import numpy as np
import random

# Définition des zones (États)
ZONES = ["Stockage", "Tri", "Emballage", "Expedition"]
n_etats = len(ZONES)

# Matrice des récompenses (R)
# Les lignes représentent l'état actuel, les colonnes l'action vers l'état suivant
# -1 indique un déplacement impossible
R = np.array([
    [-1,  0, -1, -1], # De Stockage vers Tri
    [ 0, -1,  0, -1], # De Tri vers Stockage ou Emballage
    [-1,  0, -1, 100], # De Emballage vers Tri ou Expedition
    [-1, -1,  0, 100]  # L'Expedition est l'état terminal
])

def obtenir_actions_possibles(etat_actuel):
    ligne_r = R[etat_actuel, :]
    actions_possibles = np.where(ligne_r >= 0)[0]
    return actions_possibles

Initialisation de la Q-Table

In [ ]:
# Initialisation de la table de connaissance à zéro
Q = np.zeros([n_etats, n_etats])

# Paramètres d'apprentissage
alpha = 0.7  # Taux d'apprentissage
gamma = 0.8  # Importance du futur
epsilon = 0.1 # Exploration

Cœur de l'Algorithme (Entraînement)

In [ ]:
def entrainer_agent(episodes):
    for i in range(episodes):
        # Choisir un état de départ aléatoire
        etat_courant = random.randint(0, n_etats - 1)
        
        # Tant que l'objectif n'est pas atteint (Zone 3 = Expedition)
        while etat_courant != 3:
            actions = obtenir_actions_possibles(etat_courant)
            
            # Logique Exploration vs Exploitation
            if random.uniform(0, 1) < epsilon:
                action_choisie = random.choice(actions)
            else:
                action_choisie = actions[np.argmax(Q[etat_courant, actions])]
            
            # Mise à jour de la Q-Table
            max_future_q = np.max(Q[action_choisie, :])
            recompense = R[etat_courant, action_choisie]
            
            # Formule de Bellman modifiée
            Q[etat_courant, action_choisie] = (1 - alpha) * Q[etat_courant, action_choisie] + \
                                             alpha * (recompense + gamma * max_future_q)
            
            etat_courant = action_choisie

# Lancement de l'apprentissage sur 500 itérations
entrainer_agent(500)
print("Entraînement terminé. Q-Table finale :\n", Q / np.max(Q) * 100)

```
Entraînement terminé. Q-Table finale :
 [[  0.   64.    0.    0. ]
 [  51.2   0.   80.    0. ]
 [   0.   64.    0.  100. ]
 [   0.    0.    0.    0. ]]
```

### 4. Test et Résultats

Une fois l'entraînement terminé, nous testons la capacité du robot à trouver le chemin le plus court depuis la Zone de Stockage (0).

In [ ]:
def test_deplacement(depart):
    chemin = [ZONES[depart]]
    etat_actuel = depart
    while etat_actuel != 3:
        action = np.argmax(Q[etat_actuel, :])
        chemin.append(ZONES[action])
        etat_actuel = action
    return chemin

resultat = test_deplacement(0)
print(f"Trajet optimal trouvé : {' -> '.join(resultat)}")

```
Trajet optimal trouvé : Stockage -> Tri -> Emballage -> Expedition
```

L'agent a réussi à converger vers une solution optimale. La Q-Table montre des valeurs élevées pour les transitions menant directement ou indirectement au Quai d'Expédition, prouvant que le robot a "compris" la structure de l'entrepôt.